In [3]:
from pathlib import Path
import re
import pandas as pd

# Folder containing the SHAP CSV files
input_folder = Path(".")

# File-name pattern:
filename_pattern = re.compile(
    r"^indiv_shap_values_Specificity_model_CNN2_trial_(\d+)\.csv$"
)

# Find and sort all matching files by model number
matching_files = []

for file_path in input_folder.glob("indiv_shap_values_Specificity_model_CNN2_trial_*.csv"):
    match = filename_pattern.match(file_path.name)

    if match:
        model_number = int(match.group(1))
        matching_files.append((model_number, file_path))

matching_files.sort(key=lambda item: item[0])

print(f"Detected {len(matching_files)} matching files:")
for model_number, file_path in matching_files:
    print(f"Model {model_number}: {file_path.name}")


# Process every detected CSV file
for model_number, input_path in matching_files:

    print(f"\nProcessing model {model_number}: {input_path.name}")

    # Pandas normally renames the second duplicate column:
    df = pd.read_csv(input_path)

    # Select the second occurrence of duplicated columns
    shap_df = df.loc[:, df.columns.str.endswith(".1")].copy()

    if shap_df.shape[1] == 0:
        print(
            f"Warning: No columns ending in '.1' were detected in "
            f"{input_path.name}. Skipping this file."
        )
        continue

    # Restore the original feature names
    shap_df.columns = shap_df.columns.str.replace(
        r"\.1$",
        "",
        regex=True
    )

    # Make sure values are numeric
    # Invalid entries are converted to NaN and ignored by mean()
    shap_df = shap_df.apply(pd.to_numeric, errors="coerce")

    # Importance magnitude = mean absolute SHAP value across all samples
    importance_magnitude = shap_df.abs().mean(axis=0)

    # Create output table
    importance_table = pd.DataFrame({
        "Feature": importance_magnitude.index,
        "Importance Magnitude": importance_magnitude.values
    })

    # Sort from highest to lowest importance
    importance_table = importance_table.sort_values(
        by="Importance Magnitude",
        ascending=False,
        ignore_index=True
    )

    # Output file name
    output_path = input_folder / (
        f"indiv_importance_magnitude_Specificity_model_CNN2_trial_{model_number}.csv"
    )

    importance_table.to_csv(output_path, index=False)

    print(f"Saved: {output_path.name}")

print("\nAll matching files have been processed.")

Detected 10 matching files:
Model 33: indiv_shap_values_Specificity_model_CNN2_trial_33.csv
Model 62: indiv_shap_values_Specificity_model_CNN2_trial_62.csv
Model 74: indiv_shap_values_Specificity_model_CNN2_trial_74.csv
Model 79: indiv_shap_values_Specificity_model_CNN2_trial_79.csv
Model 82: indiv_shap_values_Specificity_model_CNN2_trial_82.csv
Model 86: indiv_shap_values_Specificity_model_CNN2_trial_86.csv
Model 90: indiv_shap_values_Specificity_model_CNN2_trial_90.csv
Model 91: indiv_shap_values_Specificity_model_CNN2_trial_91.csv
Model 92: indiv_shap_values_Specificity_model_CNN2_trial_92.csv
Model 95: indiv_shap_values_Specificity_model_CNN2_trial_95.csv

Processing model 33: indiv_shap_values_Specificity_model_CNN2_trial_33.csv
Saved: indiv_importance_magnitude_Specificity_model_CNN2_trial_33.csv

Processing model 62: indiv_shap_values_Specificity_model_CNN2_trial_62.csv
Saved: indiv_importance_magnitude_Specificity_model_CNN2_trial_62.csv

Processing model 74: indiv_shap_values_